In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents) 


72

In [4]:

print(documents[0]["filename"])
print(documents[1]["filename"])
print(documents[2]["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [8]:
docCutted=documents[:3]
docCutted

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [9]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [10]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
import json

user_prompt = json.dumps(docCutted)

In [14]:
user_prompt

'[{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most commo

In [15]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [16]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [17]:
result = response.output_parsed

print(result)

questions=["What problem does RAG solve that a plain language model can't handle well?", 'Why do we need to pass course FAQ text into the prompt instead of asking the model directly?', 'What are the three main parts of a basic RAG pipeline in this lesson?', 'Why is search quality so important in a RAG system?', "What do I need installed and configured before starting this module's notebook?"]


In [18]:
print(result.questions)

["What problem does RAG solve that a plain language model can't handle well?", 'Why do we need to pass course FAQ text into the prompt instead of asking the model directly?', 'What are the three main parts of a basic RAG pipeline in this lesson?', 'Why is search quality so important in a RAG system?', "What do I need installed and configured before starting this module's notebook?"]


In [19]:
from evaluation_utils import llm_structured

In [20]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Why does the course use RAG instead of asking the LLM directly?', 'What are the main limits of a plain LLM when you ask it about your own documents or course info?', 'What do I actually need installed and set up before starting this module?', 'How does the FAQ bot answer a student question using retrieval and generation together?', 'If the retrieved documents are wrong, what happens to the final answer and why?']


In [21]:
usage.input_tokens, usage.output_tokens

(3731, 98)

In [22]:
from evaluation_utils import calc_price

In [23]:
cost = calc_price(usage)

cost

{'input_cost': 0.00279825, 'output_cost': 0.000441, 'total_cost': 0.00323925}

In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

In [33]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(docCutted):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    print(usage.input_tokens, usage.output_tokens)
    usages.append(usage)


  0%|          | 0/3 [00:00<?, ?it/s]

1020 100
1286 116
1753 99


In [34]:
print('Average input tokens:', sum(u.input_tokens for u in usages) / len(usages))

Average input tokens: 1353.0


The First Answer Question: 1400

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [6]:
len(chunks)

295

In [25]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [26]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [27]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [28]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [29]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [30]:
text_results = index.search(
    q,
    num_results = 5
)

text= [doc['filename'] for doc in text_results]
len(text)
text

['01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md']

The Second Answer is: 01-agentic-rag/lessons/03-rag.md

In [31]:
from embedder import Embedder
embed = Embedder()

In [32]:
texts = [doc['content'] for doc in chunks]
X = embed.encode_batch(texts)
X.shape

(295, 384)

In [33]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
def vector_search(query):
   query_vector = embed.encode(query)

    def vector_search(query):
        query_vector = embed.encode(query)

        return vindex.search(
            query_vector,
            num_results=5
        )

In [34]:
query_vector = embed.encode(q)
results = vindex.search(query_vector, num_results=5)

In [35]:
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

The Third Answer is: 01-agentic-rag/lessons/01-intro.md

In [57]:
def text_search(query):
    boost_dict = {"filename": 1.0, "filename": 1.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [37]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [38]:
from tqdm.auto import tqdm

In [39]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [40]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [41]:
relevance_total_text = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [46]:
relevance_total_text

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [47]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [48]:
hr=hit_rate(relevance_total_text)

In [50]:
hr

0.7583333333333333

The Forth Answer is: 0.76

In [53]:
from minsearch import VectorSearch

In [54]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)